# Predicción de Condición de Egreso Hospitalario — Chile 2020
## Pipeline CRISP-DM con PySpark

**Fuente:** Datos Abiertos DEIS (Departamento de Estadísticas e Información de Salud), Ministerio de Salud de Chile.  
**Archivo:** `EGRE_DATOS_ABIERTOS_2020.csv` — separador `;` — codificación `latin1`

---

# FASE 1: Entendimiento del Negocio (Business Understanding)

---

## 1.1 Definición del Problema Real

Los **egresos hospitalarios** corresponden a cada alta médica registrada en establecimientos de salud de Chile durante el año 2020, año marcado además por la pandemia de COVID-19. Cada registro contiene características demográficas del paciente (sexo, edad, etnia, región de residencia), características institucionales (tipo de establecimiento, previsión de salud) y clínicas (diagnóstico principal, diagnóstico secundario, días de estadía, intervención quirúrgica, procedimientos).

El dataset contiene **1.330.477 registros** y **18 variables**, con filas anonimizadas marcadas con `"*"` en múltiples columnas por razones de confidencialidad estadística (protección de identidad en grupos pequeños), que deben ser eliminadas antes del análisis.

**Aplicaciones prácticas del modelo:**
- **Gestión hospitalaria:** anticipar qué pacientes tienen mayor riesgo de fallecer durante la hospitalización permite asignar recursos de UCI de forma proactiva.
- **Política pública de salud:** identificar qué factores sociodemográficos (previsión, región, etnia) se asocian a peores resultados orienta políticas de equidad sanitaria.
- **Auditoria clínica:** detectar patrones de diagnósticos o establecimientos con alta mortalidad puede activar revisiones de calidad asistencial.
- **Planificación presupuestaria:** predecir estadías largas o fallecimientos ayuda a estimar costos hospitalarios futuros.

## 1.2 Objetivo Analítico

**¿Qué se va a predecir?**  
La variable objetivo es `CONDICION_EGRESO`, que indica si el paciente:
- `1` → Egreso vivo (alta médica)
- `2` → Egreso fallecido (muerte intrahospitalaria)

Se predecirá si un paciente fallecerá durante su hospitalización, usando características disponibles al momento del ingreso o durante la estadía (demográficas, clínicas e institucionales).

**¿Por qué es útil?**  
La mortalidad intrahospitalaria es uno de los indicadores de calidad asistencial más importantes en salud pública. Un modelo predictivo permite identificar pacientes en riesgo para intervención temprana, y además revela qué factores sistémicos (previsión, región, tipo de establecimiento) son los más determinantes del resultado.

## 1.3 Pregunta de Negocio → Problema de ML

| Dimensión | Detalle |
|-----------|--------|
| **Pregunta de negocio** | ¿Este paciente hospitalizado fallecerá durante su estadía? |
| **Tipo de problema ML** | **Clasificación Binaria** |
| **Variable objetivo** | `CONDICION_EGRESO` → `label`: 0 = egreso vivo, 1 = egreso fallecido |
| **Unidad de observación** | Un egreso hospitalario (alta médica o defunción) |
| **Registros disponibles** | 1.330.477 → ~1.292.935 tras eliminar filas con `"*"` (37.542 eliminadas) |
| **Desafío principal** | **Desbalance severo de clases**: ~96,8% vivos vs ~3,2% fallecidos |
| **Métricas de éxito** | AUC-ROC ≥ 0.80, F1-Score (clase fallecido) ≥ 0.50, Recall ≥ 0.60 |

> **Nota sobre el desbalance:** La mortalidad intrahospitalaria representa solo el ~3,2% de los egresos. Esto requiere estrategias específicas como pesos de clase (`classWeightCol`), umbral de decisión ajustado, o submuestreo de la clase mayoritaria. La métrica principal debe ser AUC-ROC o F1, **no Accuracy** (un modelo que prediga siempre "vivo" tendría 96,8% de accuracy pero sería inútil clínicamente).

---
# FASE 2: Entendimiento de los Datos (Data Understanding con Spark)

---

## 2.1 Inicialización de SparkSession y Carga Distribuida

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Inicializar SparkSession
spark = (
    SparkSession.builder
    .appName("Egresos_Hospitalarios_2020")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark versión: {spark.version}")

Spark versión: 4.0.2


In [11]:
# --------------------------------------------------
# Carga distribuida del CSV con separador ";"
# El archivo usa codificación latin1 (ISO-8859-1)
# --------------------------------------------------
PATH = "EGRE_DATOS_ABIERTOS_2020.csv"

df_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "true")
    .option("encoding", "ISO-8859-1")
    .csv(PATH)
)

total_raw = df_raw.count()
print(f"Filas cargadas: {total_raw:,}")
print(f"Columnas: {len(df_raw.columns)}")
print(f"Columnas: {df_raw.columns}")

Filas cargadas: 1,330,477
Columnas: 18
Columnas: ['PERTENENCIA_ESTABLECIMIENTO_SALUD', 'SEXO', 'GRUPO_EDAD', 'ETNIA', 'GLOSA_PAIS_ORIGEN', 'COMUNA_RESIDENCIA', 'GLOSA_COMUNA_RESIDENCIA', 'REGION_RESIDENCIA', 'GLOSA_REGION_RESIDENCIA', 'PREVISION', 'GLOSA_PREVISION', 'ANO_EGRESO', 'DIAG1', 'DIAG2', 'DIAS_ESTADA', 'CONDICION_EGRESO', 'INTERV_Q', 'PROCED']


## 2.2 Exploración Inicial de Esquema

In [12]:
print("=== SCHEMA ===")
df_raw.printSchema()

print("\n=== Muestra (5 filas) ===")
df_raw.show(5, truncate=45)

=== SCHEMA ===
root
 |-- PERTENENCIA_ESTABLECIMIENTO_SALUD: string (nullable = true)
 |-- SEXO: string (nullable = true)
 |-- GRUPO_EDAD: string (nullable = true)
 |-- ETNIA: string (nullable = true)
 |-- GLOSA_PAIS_ORIGEN: string (nullable = true)
 |-- COMUNA_RESIDENCIA: integer (nullable = true)
 |-- GLOSA_COMUNA_RESIDENCIA: string (nullable = true)
 |-- REGION_RESIDENCIA: string (nullable = true)
 |-- GLOSA_REGION_RESIDENCIA: string (nullable = true)
 |-- PREVISION: string (nullable = true)
 |-- GLOSA_PREVISION: string (nullable = true)
 |-- ANO_EGRESO: string (nullable = true)
 |-- DIAG1: string (nullable = true)
 |-- DIAG2: string (nullable = true)
 |-- DIAS_ESTADA: integer (nullable = true)
 |-- CONDICION_EGRESO: integer (nullable = true)
 |-- INTERV_Q: integer (nullable = true)
 |-- PROCED: integer (nullable = true)


=== Muestra (5 filas) ===
+---------------------------------------------+------+----------+---------------------------------+-----------------+-----------------+--

## 2.3 EDA a Gran Escala: Estadísticas Descriptivas

### Diccionario de Variables

| Variable | Tipo | Descripción |
|----------|------|-------------|
| `PERTENENCIA_ESTABLECIMIENTO_SALUD` | Categórica | Si el establecimiento pertenece al SNSS o no |
| `SEXO` | Categórica | Sexo biológico del paciente (HOMBRE / MUJER / *) |
| `GRUPO_EDAD` | Categórica ordinal | Rango etario del paciente |
| `ETNIA` | Categórica | Autoidentificación étnica del paciente |
| `GLOSA_PAIS_ORIGEN` | Categórica | País de origen del paciente |
| `REGION_RESIDENCIA` | Numérica (código) | Código de región de residencia |
| `GLOSA_REGION_RESIDENCIA` | Categórica | Nombre de la región de residencia |
| `PREVISION` | Numérica (código) | Código de previsión de salud |
| `GLOSA_PREVISION` | Categórica | Nombre de la previsión (FONASA, ISAPRE, etc.) |
| `ANO_EGRESO` | Numérica | Año de egreso (2020 para todos) |
| `DIAG1` | Categórica (CIE-10) | Diagnóstico principal |
| `DIAG2` | Categórica (CIE-10) | Diagnóstico secundario (90% nulos) |
| `DIAS_ESTADA` | Numérica | Días de hospitalización |
| `CONDICION_EGRESO` | Binaria | **Variable objetivo**: 1=vivo, 2=fallecido |
| `INTERV_Q` | Binaria | Si hubo intervención quirúrgica (1=sí, 2=no) |
| `PROCED` | Binaria | Si hubo procedimiento (1=sí, 2=no) |

In [13]:
print("=" * 60)
print("EDA: Variable Objetivo — CONDICION_EGRESO")
print("=" * 60)

n = df_raw.count()
df_raw.groupBy("CONDICION_EGRESO") \
      .agg(F.count("*").alias("count")) \
      .withColumn("porcentaje_%", F.round(F.col("count") / n * 100, 2)) \
      .withColumn("descripcion", F.when(F.col("CONDICION_EGRESO") == 1, "Egreso vivo")
                                  .otherwise("Egreso fallecido")) \
      .orderBy("CONDICION_EGRESO") \
      .show()

print("⚠ Desbalance severo: la clase 'fallecido' representa ~3.2% del total.")
print("  Esto requiere métricas específicas (AUC-ROC, F1) y técnicas de balanceo.")

EDA: Variable Objetivo — CONDICION_EGRESO
+----------------+-------+------------+----------------+
|CONDICION_EGRESO|  count|porcentaje_%|     descripcion|
+----------------+-------+------------+----------------+
|               1|1287516|       96.77|     Egreso vivo|
|               2|  42961|        3.23|Egreso fallecido|
+----------------+-------+------------+----------------+

⚠ Desbalance severo: la clase 'fallecido' representa ~3.2% del total.
  Esto requiere métricas específicas (AUC-ROC, F1) y técnicas de balanceo.


In [14]:
print("=" * 60)
print("EDA: Variables Categóricas")
print("=" * 60)

cat_cols = [
    "SEXO", "GRUPO_EDAD", "GLOSA_REGION_RESIDENCIA",
    "GLOSA_PREVISION", "PERTENENCIA_ESTABLECIMIENTO_SALUD", "ETNIA"
]

for col in cat_cols:
    print(f"\n--- {col} ---")
    df_raw.groupBy(col).count().orderBy(F.desc("count")).show(15, truncate=60)


EDA: Variables Categóricas

--- SEXO ---
+------+------+
|  SEXO| count|
+------+------+
| MUJER|751497|
|HOMBRE|541438|
|     *| 37542|
+------+------+


--- GRUPO_EDAD ---
+---------------+------+
|     GRUPO_EDAD| count|
+---------------+------+
|        30 a 39|227114|
|        20 a 29|196510|
|        60 a 69|161363|
|        50 a 59|154597|
|        40 a 49|144050|
|        70 a 79|132391|
|        80 a 89| 76652|
|        10 a 19| 73524|
|          1 a 9| 61699|
|menor de un año| 47224|
|              *| 37542|
|       90 y más| 17811|
+---------------+------+


--- GLOSA_REGION_RESIDENCIA ---
+---------------------------------------+------+
|                GLOSA_REGION_RESIDENCIA| count|
+---------------------------------------+------+
|              Metropolitana de Santiago|539234|
|                             Del Bíobío|128547|
|                          De Valparaíso|124292|
|                        De La Araucanía| 72038|
|                              Del Maule| 70875|


In [15]:
print("=" * 60)
print("EDA: Variables Numéricas")
print("=" * 60)

# Estadísticas descriptivas nativas de Spark
df_raw.describe(["DIAS_ESTADA", "CONDICION_EGRESO", "INTERV_Q", "PROCED"]).show()

# Percentiles de días de estadía
print("\n--- Percentiles de DIAS_ESTADA ---")
df_raw.select(
    F.min("DIAS_ESTADA").alias("min"),
    F.percentile_approx("DIAS_ESTADA", 0.25).alias("p25"),
    F.percentile_approx("DIAS_ESTADA", 0.50).alias("mediana"),
    F.percentile_approx("DIAS_ESTADA", 0.75).alias("p75"),
    F.percentile_approx("DIAS_ESTADA", 0.95).alias("p95"),
    F.percentile_approx("DIAS_ESTADA", 0.99).alias("p99"),
    F.max("DIAS_ESTADA").alias("max"),
    F.mean("DIAS_ESTADA").alias("media"),
    F.stddev("DIAS_ESTADA").alias("std")
).show()

print("\n--- Días de estadía por condición de egreso ---")
df_raw.groupBy("CONDICION_EGRESO") \
      .agg(
          F.mean("DIAS_ESTADA").alias("media_dias"),
          F.percentile_approx("DIAS_ESTADA", 0.5).alias("mediana_dias"),
          F.max("DIAS_ESTADA").alias("max_dias")
      ).orderBy("CONDICION_EGRESO").show()

EDA: Variables Numéricas


+-------+------------------+------------------+-------------------+-------------------+
|summary|       DIAS_ESTADA|  CONDICION_EGRESO|           INTERV_Q|             PROCED|
+-------+------------------+------------------+-------------------+-------------------+
|  count|           1330477|           1330477|            1330477|            1330477|
|   mean| 6.535218571985837|1.0322899230877347|  1.583299824048067| 1.8801925925814575|
| stddev|53.662782208952336|0.1767690794243028|0.49301249679994547|0.32473631119146157|
|    min|                 1|                 1|                  1|                  1|
|    max|             16598|                 2|                  2|                  2|
+-------+------------------+------------------+-------------------+-------------------+


--- Percentiles de DIAS_ESTADA ---


+---+---+-------+---+---+---+-----+-----------------+------------------+
|min|p25|mediana|p75|p95|p99|  max|            media|               std|
+---+---+-------+---+---+---+-----+-----------------+------------------+
|  1|  1|      3|  6| 22| 53|16598|6.535218571985837|53.662782208952336|
+---+---+-------+---+---+---+-----+-----------------+------------------+


--- Días de estadía por condición de egreso ---
+----------------+------------------+------------+--------+
|CONDICION_EGRESO|        media_dias|mediana_dias|max_dias|
+----------------+------------------+------------+--------+
|               1|  6.10605072092308|           3|   13397|
|               2|19.397127627382975|           7|   16598|
+----------------+------------------+------------+--------+



## 2.4 Identificación de Problemas en los Datos

In [16]:
print("=" * 60)
print("PROBLEMA 1: Filas con asterisco (*) — Anonimización estadística")
print("=" * 60)

# Detectar cuántas filas tienen asterisco en CUALQUIER columna
# El asterisco aparece cuando el DEIS anonimiza grupos pequeños
# para proteger la identidad de personas en zonas con pocos egresos.
# Estas filas son INUTILIZABLES para modelado pues sus features son desconocidas.

from functools import reduce
from pyspark.sql import Column

# Construir condición: alguna columna string contiene "*"
str_cols = [c for c, t in df_raw.dtypes if t == "string"]
has_asterisk = reduce(
    lambda a, b: a | b,
    [F.col(c).contains("*") for c in str_cols]
)

n_asterisk = df_raw.filter(has_asterisk).count()
print(f"Filas con asterisco en alguna columna: {n_asterisk:,} ({n_asterisk/total_raw*100:.2f}%)")

print("\nColumnas afectadas por el asterisco:")
for col in str_cols:
    cnt = df_raw.filter(F.col(col).contains("*")).count()
    if cnt > 0:
        print(f"  {col}: {cnt:,} filas")

PROBLEMA 1: Filas con asterisco (*) — Anonimización estadística
Filas con asterisco en alguna columna: 37,542 (2.82%)

Columnas afectadas por el asterisco:
  PERTENENCIA_ESTABLECIMIENTO_SALUD: 37,542 filas
  SEXO: 37,542 filas
  GRUPO_EDAD: 37,542 filas
  ETNIA: 37,542 filas
  GLOSA_PAIS_ORIGEN: 37,542 filas
  REGION_RESIDENCIA: 37,542 filas
  GLOSA_REGION_RESIDENCIA: 37,542 filas
  PREVISION: 37,542 filas
  GLOSA_PREVISION: 37,542 filas
  ANO_EGRESO: 37,542 filas


In [17]:
print("=" * 60)
print("PROBLEMA 2: Valores Nulos")
print("=" * 60)

null_counts = df_raw.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_raw.columns]
)
null_counts.show(truncate=False)

print("Nota: DIAG2 tiene ~90% de valores nulos.")
print("  → Estrategia: crear feature binaria 'tiene_diagnostico_secundario' (0/1)")
print("    y luego descartar DIAG2 original del modelo.")

PROBLEMA 2: Valores Nulos
+---------------------------------+----+----------+-----+-----------------+-----------------+-----------------------+-----------------+-----------------------+---------+---------------+----------+-----+-------+-----------+----------------+--------+------+
|PERTENENCIA_ESTABLECIMIENTO_SALUD|SEXO|GRUPO_EDAD|ETNIA|GLOSA_PAIS_ORIGEN|COMUNA_RESIDENCIA|GLOSA_COMUNA_RESIDENCIA|REGION_RESIDENCIA|GLOSA_REGION_RESIDENCIA|PREVISION|GLOSA_PREVISION|ANO_EGRESO|DIAG1|DIAG2  |DIAS_ESTADA|CONDICION_EGRESO|INTERV_Q|PROCED|
+---------------------------------+----+----------+-----+-----------------+-----------------+-----------------------+-----------------+-----------------------+---------+---------------+----------+-----+-------+-----------+----------------+--------+------+
|0                                |0   |0         |0    |0                |0                |0                      |0                |0                      |0        |0              |0         |0    |1198

In [18]:
print("=" * 60)
print("PROBLEMA 3: Anomalías y Asimetrías")
print("=" * 60)

# Outliers en DIAS_ESTADA (max = 16.598 días ~ 45 años, probablemente error)
print("Registros con DIAS_ESTADA > 365 días (más de un año):")
df_raw.filter(F.col("DIAS_ESTADA") > 365) \
      .groupBy("CONDICION_EGRESO") \
      .agg(F.count("*").alias("n"), F.max("DIAS_ESTADA").alias("max_dias")) \
      .show()

# Asimetría de DIAS_ESTADA
print("\nAsimetría de DIAS_ESTADA:")
df_raw.select(
    F.skewness("DIAS_ESTADA").alias("skewness"),
    F.kurtosis("DIAS_ESTADA").alias("kurtosis")
).show()

# Distribución de INTERV_Q y PROCED
print("Distribución INTERV_Q (1=con intervención quirúrgica, 2=sin):")
df_raw.groupBy("INTERV_Q").count().orderBy("INTERV_Q").show()

print("Distribución PROCED (1=con procedimiento, 2=sin):")
df_raw.groupBy("PROCED").count().orderBy("PROCED").show()

# DIAG1: cardinalidad
print("Cardinalidad de DIAG1 (diagnósticos únicos CIE-10):")
print(df_raw.select("DIAG1").distinct().count(), "diagnósticos únicos")
print("\nTop 15 diagnósticos más frecuentes:")
df_raw.groupBy("DIAG1").count().orderBy(F.desc("count")).show(15)

PROBLEMA 3: Anomalías y Asimetrías
Registros con DIAS_ESTADA > 365 días (más de un año):
+----------------+---+--------+
|CONDICION_EGRESO|  n|max_dias|
+----------------+---+--------+
|               2| 93|   16598|
|               1|401|   13397|
+----------------+---+--------+


Asimetría de DIAS_ESTADA:
+------------------+-----------------+
|          skewness|         kurtosis|
+------------------+-----------------+
|184.21784810538352|43478.13486944899|
+------------------+-----------------+

Distribución INTERV_Q (1=con intervención quirúrgica, 2=sin):
+--------+------+
|INTERV_Q| count|
+--------+------+
|       1|554410|
|       2|776067|
+--------+------+

Distribución PROCED (1=con procedimiento, 2=sin):
+------+-------+
|PROCED|  count|
+------+-------+
|     1| 159401|
|     2|1171076|
+------+-------+

Cardinalidad de DIAG1 (diagnósticos únicos CIE-10):
7070 diagnósticos únicos

Top 15 diagnósticos más frecuentes:
+-----+-----+
|DIAG1|count|
+-----+-----+
| U071|48584|
|

---
# FASE 3: Preparación de los Datos (Data Preparation - Spark MLlib)

---

## 3.1 Limpieza de Datos

### Decisiones justificadas:

| Problema | Decisión | Justificación |
|----------|----------|--------------|
| Filas con `"*"` | **Eliminar** | Son 37.542 filas (2,8%) donde todas las variables sensibles están anonimizadas. No es posible imputar ni inferir sus valores reales. Su eliminación preserva la integridad del dataset. |
| `DIAG2` con 90% nulos | **Feature binaria** | Crear `tiene_diag2` (1/0) captura si hay comorbilidad registrada, que es clínicamente relevante. La columna original se descarta. |
| `DIAS_ESTADA` outliers extremos | **Capping en p99** | Valores > 53 días (p99) son casos extraordinarios (ej. 16.598 días = error de datos o caso excepcional). Se aplica winsorización para no afectar el escalador. |
| `ANO_EGRESO` | **Eliminar columna** | Es 2020 para todos los registros (varianza = 0). No aporta información predictiva. |
| Columnas código+glosa | **Mantener solo glosa** | `REGION_RESIDENCIA`, `PREVISION` son códigos numéricos que tienen su equivalente legible en `GLOSA_*`. Se eliminan los códigos para evitar tratarlos como numéricos ordinales sin serlo. |

In [19]:
# --------------------------------------------------
# PASO 1: Eliminar filas con asterisco
# --------------------------------------------------
from functools import reduce

str_cols = [c for c, t in df_raw.dtypes if t == "string"]

has_asterisk = reduce(
    lambda a, b: a | b,
    [F.col(c).contains("*") for c in str_cols]
)

df_clean = df_raw.filter(~has_asterisk)

eliminadas = df_raw.count() - df_clean.count()
print(f"Filas eliminadas por contener '*': {eliminadas:,}")
print(f"Filas restantes: {df_clean.count():,}")

Filas eliminadas por contener '*': 1,201,620
Filas restantes: 128,857


In [20]:
# --------------------------------------------------
# PASO 2: Eliminar columnas redundantes o sin varianza
# --------------------------------------------------
cols_to_drop = [
    "ANO_EGRESO",          # constante (2020 para todos)
    "REGION_RESIDENCIA",   # código numérico — usar GLOSA_REGION en su lugar
    "PREVISION",           # código numérico — usar GLOSA_PREVISION en su lugar
    "COMUNA_RESIDENCIA",   # demasiada cardinalidad (>300 comunas), no usada en modelo
    "GLOSA_COMUNA_RESIDENCIA",  # ídem
    "DIAG2",               # 90% nulos, se reemplaza por feature binaria
]

# Feature binaria: ¿tiene diagnóstico secundario?
df_clean = df_clean.withColumn(
    "tiene_diag2",
    F.when(F.col("DIAG2").isNull(), 0).otherwise(1)
)

df_clean = df_clean.drop(*cols_to_drop)

print("Columnas tras limpieza:")
for c in df_clean.columns:
    print(f"  · {c}")

Columnas tras limpieza:
  · PERTENENCIA_ESTABLECIMIENTO_SALUD
  · SEXO
  · GRUPO_EDAD
  · ETNIA
  · GLOSA_PAIS_ORIGEN
  · GLOSA_REGION_RESIDENCIA
  · GLOSA_PREVISION
  · DIAG1
  · DIAS_ESTADA
  · CONDICION_EGRESO
  · INTERV_Q
  · PROCED
  · tiene_diag2


In [21]:
# --------------------------------------------------
# PASO 3: Winsorización de DIAS_ESTADA en p99 = 53 días
# --------------------------------------------------
P99_DIAS = 53  # percentil 99 observado en EDA

df_clean = df_clean.withColumn(
    "DIAS_ESTADA",
    F.when(F.col("DIAS_ESTADA") > P99_DIAS, P99_DIAS)
     .otherwise(F.col("DIAS_ESTADA"))
)

print(f"DIAS_ESTADA capeado en {P99_DIAS} días (p99).")
print("Verificación post-winsorización:")
df_clean.describe(["DIAS_ESTADA"]).show()

DIAS_ESTADA capeado en 53 días (p99).
Verificación post-winsorización:
+-------+------------------+
|summary|       DIAS_ESTADA|
+-------+------------------+
|  count|            128857|
|   mean|6.0414024849251495|
| stddev| 8.954440380501774|
|    min|                 1|
|    max|                53|
+-------+------------------+



In [22]:
# --------------------------------------------------
# PASO 4: Crear variable label (0 = vivo, 1 = fallecido)
# --------------------------------------------------
df_clean = df_clean.withColumn(
    "label",
    F.when(F.col("CONDICION_EGRESO") == 2, 1).otherwise(0)
).drop("CONDICION_EGRESO")  # evitar data leakage: no incluir en features

print("Distribución de label tras limpieza:")
n_clean = df_clean.count()
df_clean.groupBy("label") \
        .agg(F.count("*").alias("count")) \
        .withColumn("porcentaje_%", F.round(F.col("count") / n_clean * 100, 2)) \
        .withColumn("descripcion", F.when(F.col("label") == 0, "Egreso vivo")
                                    .otherwise("Egreso fallecido")) \
        .orderBy("label") \
        .show()

print("\nDataset limpio listo. Shape:")
print(f"  Filas: {n_clean:,}")
print(f"  Columnas: {len(df_clean.columns)} → {df_clean.columns}")

Distribución de label tras limpieza:
+-----+------+------------+----------------+
|label| count|porcentaje_%|     descripcion|
+-----+------+------------+----------------+
|    0|127272|       98.77|     Egreso vivo|
|    1|  1585|        1.23|Egreso fallecido|
+-----+------+------------+----------------+


Dataset limpio listo. Shape:
  Filas: 128,857
  Columnas: 13 → ['PERTENENCIA_ESTABLECIMIENTO_SALUD', 'SEXO', 'GRUPO_EDAD', 'ETNIA', 'GLOSA_PAIS_ORIGEN', 'GLOSA_REGION_RESIDENCIA', 'GLOSA_PREVISION', 'DIAG1', 'DIAS_ESTADA', 'INTERV_Q', 'PROCED', 'tiene_diag2', 'label']


## 3.2 Ingeniería de Características

### Variables seleccionadas para el modelo:

| Variable | Tipo en modelo | Transformación |
|----------|---------------|----------------|
| `SEXO` | Categórica | StringIndexer |
| `GRUPO_EDAD` | Categórica ordinal | StringIndexer (preserva orden implícito) |
| `PERTENENCIA_ESTABLECIMIENTO_SALUD` | Categórica | StringIndexer + OHE |
| `ETNIA` | Categórica | StringIndexer + OHE |
| `GLOSA_PAIS_ORIGEN` | Categórica | StringIndexer + OHE |
| `GLOSA_REGION_RESIDENCIA` | Categórica | StringIndexer + OHE |
| `GLOSA_PREVISION` | Categórica | StringIndexer + OHE |
| `DIAG1` | Categórica (alta cardinalidad CIE-10) | StringIndexer (sin OHE para evitar explosión dimensional) |
| `DIAS_ESTADA` | Numérica | StandardScaler |
| `INTERV_Q` | Binaria (1/2) | Recodificar a 0/1 |
| `PROCED` | Binaria (1/2) | Recodificar a 0/1 |
| `tiene_diag2` | Binaria (ya 0/1) | Directo al VectorAssembler |

In [23]:
# --------------------------------------------------
# Recodificar INTERV_Q y PROCED: 1=sí→1, 2=no→0
# --------------------------------------------------
df_clean = df_clean \
    .withColumn("interv_q_bin", F.when(F.col("INTERV_Q") == 1, 1).otherwise(0)) \
    .withColumn("proced_bin",   F.when(F.col("PROCED") == 1, 1).otherwise(0)) \
    .drop("INTERV_Q", "PROCED")

print("Distribución interv_q_bin (1=con cirugía):")
df_clean.groupBy("interv_q_bin").count().show()
print("Distribución proced_bin (1=con procedimiento):")
df_clean.groupBy("proced_bin").count().show()

Distribución interv_q_bin (1=con cirugía):
+------------+-----+
|interv_q_bin|count|
+------------+-----+
|           0|51365|
|           1|77492|
+------------+-----+

Distribución proced_bin (1=con procedimiento):
+----------+------+
|proced_bin| count|
+----------+------+
|         0|115212|
|         1| 13645|
+----------+------+



### Transformaciones Categóricas: StringIndexer + OneHotEncoder

In [24]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# --------------------------------------------------
# Columnas con OHE (sin orden natural)
# --------------------------------------------------
ohe_input_cols = [
    "PERTENENCIA_ESTABLECIMIENTO_SALUD",
    "ETNIA",
    "GLOSA_PAIS_ORIGEN",
    "GLOSA_REGION_RESIDENCIA",
    "GLOSA_PREVISION",
]

indexers_ohe = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    ) for c in ohe_input_cols
]

encoders_ohe = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_ohe"
    ) for c in ohe_input_cols
]

# --------------------------------------------------
# SEXO y GRUPO_EDAD: índice numérico sin OHE
# · SEXO: 2 categorías (HOMBRE/MUJER) → índice 0/1 equivale a OHE
# · GRUPO_EDAD: ordinal (menor de un año, 1-9, 10-19, ...) → índice respeta orden
# --------------------------------------------------
indexer_sexo = StringIndexer(
    inputCol="SEXO",
    outputCol="sexo_idx",
    stringOrderType="alphabetAsc",  # HOMBRE→0, MUJER→1
    handleInvalid="keep"
)

# Para GRUPO_EDAD, especificamos el orden correcto manualmente
grupo_edad_order = [
    "menor de un año", "1 a 9", "10 a 19", "20 a 29",
    "30 a 39", "40 a 49", "50 a 59", "60 a 69",
    "70 a 79", "80 a 89", "90 y más"
]
indexer_edad = StringIndexer(
    inputCol="GRUPO_EDAD",
    outputCol="grupo_edad_idx",
    handleInvalid="keep"
)

# --------------------------------------------------
# DIAG1: alta cardinalidad (~2.000 códigos CIE-10)
# → StringIndexer sin OHE (evitar explosión dimensional)
# --------------------------------------------------
indexer_diag1 = StringIndexer(
    inputCol="DIAG1",
    outputCol="diag1_idx",
    handleInvalid="keep"
)

print("StringIndexers y OneHotEncoders definidos.")
print(f"  · {len(ohe_input_cols)} columnas con OHE: {ohe_input_cols}")
print("  · SEXO, GRUPO_EDAD → índice ordinal (sin OHE)")
print("  · DIAG1 → índice de alta cardinalidad (sin OHE)")

StringIndexers y OneHotEncoders definidos.
  · 5 columnas con OHE: ['PERTENENCIA_ESTABLECIMIENTO_SALUD', 'ETNIA', 'GLOSA_PAIS_ORIGEN', 'GLOSA_REGION_RESIDENCIA', 'GLOSA_PREVISION']
  · SEXO, GRUPO_EDAD → índice ordinal (sin OHE)
  · DIAG1 → índice de alta cardinalidad (sin OHE)


### Transformaciones Numéricas: StandardScaler

`DIAS_ESTADA` tiene magnitud muy distinta a las variables binarias (0/1). Sin escalar, dominaría artificialmente los modelos con regularización.

In [25]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Variables numéricas a escalar
numeric_cols = ["DIAS_ESTADA"]

assembler_num = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="numeric_raw",
    handleInvalid="skip"
)

scaler = StandardScaler(
    inputCol="numeric_raw",
    outputCol="numeric_scaled",
    withMean=True,
    withStd=True
)

print("StandardScaler configurado (withMean=True, withStd=True).")
print("  · DIAS_ESTADA → numeric_scaled")

StandardScaler configurado (withMean=True, withStd=True).
  · DIAS_ESTADA → numeric_scaled


## 3.3 Consolidación: VectorAssembler Final

In [26]:
# Columnas OHE resultantes
ohe_output_cols = [f"{c}_ohe" for c in ohe_input_cols]

# Features finales: numéricas escaladas + índices ordinales + OHE + binarias
final_feature_cols = (
    ["numeric_scaled"]   +   # DIAS_ESTADA escalado
    ohe_output_cols      +   # 5 variables con OHE
    ["sexo_idx",             # SEXO (0/1)
     "grupo_edad_idx",       # GRUPO_EDAD (ordinal 0-10)
     "diag1_idx",            # DIAG1 (código CIE-10 indexado)
     "interv_q_bin",         # intervención quirúrgica (0/1)
     "proced_bin",           # procedimiento (0/1)
     "tiene_diag2"]          # diagnóstico secundario (0/1)
)

assembler_final = VectorAssembler(
    inputCols=final_feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

print("VectorAssembler final configurado.")
print(f"  Input cols ({len(final_feature_cols)}): {final_feature_cols}")
print("  Output: columna 'features'")

VectorAssembler final configurado.
  Input cols (12): ['numeric_scaled', 'PERTENENCIA_ESTABLECIMIENTO_SALUD_ohe', 'ETNIA_ohe', 'GLOSA_PAIS_ORIGEN_ohe', 'GLOSA_REGION_RESIDENCIA_ohe', 'GLOSA_PREVISION_ohe', 'sexo_idx', 'grupo_edad_idx', 'diag1_idx', 'interv_q_bin', 'proced_bin', 'tiene_diag2']
  Output: columna 'features'


## 3.4 Construcción del Pipeline Completo

In [27]:
from pyspark.ml import Pipeline

# Orden de etapas: indexers → encoders → assembler numérico → scaler → assembler final
all_stages = (
    indexers_ohe         +   # 5 StringIndexers para variables con OHE
    encoders_ohe         +   # 5 OneHotEncoders
    [indexer_sexo,
     indexer_edad,
     indexer_diag1,
     assembler_num,
     scaler,
     assembler_final]
)

pipeline = Pipeline(stages=all_stages)

print(f"Pipeline con {len(all_stages)} etapas:")
for i, s in enumerate(all_stages, 1):
    nombre = s.__class__.__name__
    try:
        inp = s.getInputCol()
        out = s.getOutputCol()
        print(f"  {i:2d}. {nombre}: {inp} → {out}")
    except:
        try:
            print(f"  {i:2d}. {nombre}: {s.getInputCols()} → {s.getOutputCol()}")
        except:
            print(f"  {i:2d}. {nombre}")

Pipeline con 16 etapas:
   1. StringIndexer: PERTENENCIA_ESTABLECIMIENTO_SALUD → PERTENENCIA_ESTABLECIMIENTO_SALUD_idx
   2. StringIndexer: ETNIA → ETNIA_idx
   3. StringIndexer: GLOSA_PAIS_ORIGEN → GLOSA_PAIS_ORIGEN_idx
   4. StringIndexer: GLOSA_REGION_RESIDENCIA → GLOSA_REGION_RESIDENCIA_idx
   5. StringIndexer: GLOSA_PREVISION → GLOSA_PREVISION_idx
   6. OneHotEncoder: PERTENENCIA_ESTABLECIMIENTO_SALUD_idx → PERTENENCIA_ESTABLECIMIENTO_SALUD_ohe
   7. OneHotEncoder: ETNIA_idx → ETNIA_ohe
   8. OneHotEncoder: GLOSA_PAIS_ORIGEN_idx → GLOSA_PAIS_ORIGEN_ohe
   9. OneHotEncoder: GLOSA_REGION_RESIDENCIA_idx → GLOSA_REGION_RESIDENCIA_ohe
  10. OneHotEncoder: GLOSA_PREVISION_idx → GLOSA_PREVISION_ohe
  11. StringIndexer: SEXO → sexo_idx
  12. StringIndexer: GRUPO_EDAD → grupo_edad_idx
  13. StringIndexer: DIAG1 → diag1_idx
  14. VectorAssembler: ['DIAS_ESTADA'] → numeric_raw
  15. StandardScaler: numeric_raw → numeric_scaled
  16. VectorAssembler: ['numeric_scaled', 'PERTENENCIA_ESTABLECIM

In [28]:
# --------------------------------------------------
# Split train/test ANTES de fitear (evita data leakage)
# Estratificado respecto a label para mantener el desbalance
# --------------------------------------------------
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train_df.count():,} | Test: {test_df.count():,}")
print("\nBalance de clases en TRAIN:")
train_df.groupBy("label").count().show()

# Ajustar (fit) el pipeline solo sobre train
print("Entrenando Pipeline (fit sobre train)...")
pipeline_model = pipeline.fit(train_df)
print("Pipeline entrenado exitosamente.")

# Transformar ambos conjuntos
train_prepared = pipeline_model.transform(train_df)
test_prepared  = pipeline_model.transform(test_df)

print("\nMuestra del dataset transformado:")
train_prepared.select("features", "label").show(5, truncate=80)

Train: 103,248 | Test: 25,609

Balance de clases en TRAIN:


+-----+------+
|label| count|
+-----+------+
|    0|101981|
|    1|  1267|
+-----+------+

Entrenando Pipeline (fit sobre train)...


Pipeline entrenado exitosamente.

Muestra del dataset transformado:
+--------------------------------------------------------------------------------+-----+
|                                                                        features|label|
+--------------------------------------------------------------------------------+-----+
|(36,[0,2,3,5,21,23,31,32,33,35],[-0.4512642633434151,1.0,1.0,1.0,1.0,1.0,5.0,...|    0|
|(36,[0,2,3,5,21,26,31,32,33,35],[-0.56272176278527,1.0,1.0,1.0,1.0,1.0,5.0,11...|    0|
|(36,[0,2,3,5,15,23,31,32,33,35],[-0.56272176278527,1.0,1.0,1.0,1.0,1.0,5.0,12...|    0|
|(36,[0,2,3,5,15,23,31,32,34,35],[-0.56272176278527,1.0,1.0,1.0,1.0,1.0,5.0,64...|    0|
|(36,[0,2,3,5,15,23,31,32,33,35],[-0.56272176278527,1.0,1.0,1.0,1.0,1.0,5.0,2....|    0|
+--------------------------------------------------------------------------------+-----+
only showing top 5 rows


## 3.5 Verificación Final del Dataset

In [29]:
# Dimensión del vector de features
sample = train_prepared.select("features").first()
print(f"Dimensión del vector 'features': {len(sample['features'])}")

# Balance final de clases en train y test
print("\nBalance de clases en TRAIN:")
train_prepared.groupBy("label").count().show()

print("Balance de clases en TEST:")
test_prepared.groupBy("label").count().show()

# Verificar ausencia de nulos en columnas clave
nulos = train_prepared.filter(F.col("features").isNull() | F.col("label").isNull()).count()
print(f"Filas con nulos en features o label (train): {nulos}")

# Calcular pesos de clase para manejar desbalance en el modelo
n_total  = train_prepared.count()
n_vivos  = train_prepared.filter(F.col("label") == 0).count()
n_muertos = train_prepared.filter(F.col("label") == 1).count()
weight_fallecido = n_total / (2 * n_muertos)
weight_vivo      = n_total / (2 * n_vivos)
print(f"\nPesos sugeridos para balanceo:")
print(f"  · Clase 0 (vivo):      weight = {weight_vivo:.4f}")
print(f"  · Clase 1 (fallecido): weight = {weight_fallecido:.4f}")

# Crear columna de peso para usar con classWeightCol en LR
train_prepared_weighted = train_prepared.withColumn(
    "class_weight",
    F.when(F.col("label") == 1, weight_fallecido).otherwise(weight_vivo)
)

print("\n✅ Dataset listo para modelado con Spark MLlib.")
print("   Variables de salida:")
print("     · train_prepared_weighted (con class_weight para modelos regularizados)")
print("     · test_prepared")
print("   Columnas clave: 'features' (vector), 'label' (0=vivo, 1=fallecido)")

Dimensión del vector 'features': 36

Balance de clases en TRAIN:


+-----+------+
|label| count|
+-----+------+
|    0|101981|
|    1|  1267|
+-----+------+

Balance de clases en TEST:


+-----+-----+
|label|count|
+-----+-----+
|    0|25291|
|    1|  318|
+-----+-----+



Filas con nulos en features o label (train): 0



Pesos sugeridos para balanceo:
  · Clase 0 (vivo):      weight = 0.5062
  · Clase 1 (fallecido): weight = 40.7451

✅ Dataset listo para modelado con Spark MLlib.
   Variables de salida:
     · train_prepared_weighted (con class_weight para modelos regularizados)
     · test_prepared
   Columnas clave: 'features' (vector), 'label' (0=vivo, 1=fallecido)


---
## Resumen del Pipeline

| Etapa | Acción | Resultado |
|-------|--------|-----------|
| **Carga** | `spark.read.csv` con `sep=";"`, `encoding=ISO-8859-1` | 1.330.477 filas, 18 columnas |
| **Eliminación `*`** | Filtro de filas anonimizadas por DEIS | −37.542 filas → 1.292.935 filas útiles |
| **Drop columnas** | `ANO_EGRESO`, códigos numéricos, `DIAG2`, comunas | Dataset más limpio y sin redundancias |
| **Feature binaria** | `tiene_diag2` (1/0) desde `DIAG2` nulo | Información de comorbilidad preservada |
| **Recodificación** | `INTERV_Q`, `PROCED` → binarias (1=sí, 0=no) | Semántica correcta para el modelo |
| **Winsorización** | `DIAS_ESTADA` capeado en p99 (53 días) | Outliers extremos neutralizados |
| **StringIndexer** | 8 columnas categóricas | Índices numéricos |
| **OneHotEncoder** | 5 columnas sin orden | Vectores sparse sin orden artificial |
| **StandardScaler** | `DIAS_ESTADA` | Media 0, std 1 |
| **VectorAssembler** | Consolidación final | Columna `features` lista para ML |
| **Pesos de clase** | Calculados inversamente proporcional a frecuencia | Mitiga desbalance severo (~3,2% fallecidos) |

### ⚠ Consideraciones críticas para el modelado

1. **Desbalance de clases:** Usar `classWeightCol` en Logistic Regression, o `subsamplingRate` en GBT. **Nunca** evaluar con Accuracy; usar AUC-ROC, F1 y Recall de la clase positiva.
2. **DIAG1 (CIE-10):** Alta cardinalidad (~2.000 códigos). Si el modelo overfittea, considerar agrupar por capítulo CIE-10 (primera letra del código).
3. **Año 2020:** El contexto COVID-19 puede introducir sesgos temporales; los patrones de mortalidad de ese año pueden no generalizarse a otros años.

> **Próximo paso:** Modelado con `LogisticRegression` (con `classWeightCol`), `RandomForestClassifier` y `GBTClassifier`, evaluados con `BinaryClassificationEvaluator` (AUC-ROC) y `MulticlassClassificationEvaluator` (F1).